# Stage 5: The targets (what we are actually predicting)

Concept → intuition → maths → real data. Day-level numbers come from `reports/stage5/days_*.parquet`
(every 7th day, 53 days per perp, produced by `scripts/stage5_analyze.py`).

In [1]:
import sys, glob
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, polars as pl
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from hft500 import align as A, inventory as inv, targets as T
pl.Config.set_tbl_width_chars(160); pl.Config.set_float_precision(3)
D = Path.cwd().parent.parent
chosen, _ = inv.pick_archives(sorted(D.glob("*.tar")))
R = A.ArchiveRoot(list(chosen.values()), fallback=A.DirRoot(D / "data"))
days = pl.concat([pl.read_parquet(f) for f in glob.glob("../reports/stage5/days_*.parquet")], how="diagonal_relaxed")
INST, DAY = "BTCUSDT_PERP", "2026-04-06"

## 1. Log returns, and why they add up

$$y_H(t)=\ln\frac{M(t+H)}{M(t)},\qquad M(t)=\text{mid}(t+10\,\text{ms})$$

Log returns **add**: $\ln\frac{M(t+2)}{M(t)}=\ln\frac{M(t+1)}{M(t)}+\ln\frac{M(t+2)}{M(t+1)}$.
So if all three targets come from one mid series, then `y_2s(t)` must equal the sum of 4 consecutive
`y_500ms`, and `y_8s(t)` the sum of 16. Check it on the supplied labels:

In [2]:
tg = R.read("targets", DAY, INST).sort("ts")
print("max |y_2s - sum of 4 y_500ms|  =", T.additivity_error(tg)["y_2s"])
print("max |y_8s - sum of 16 y_500ms| =", T.additivity_error(tg)["y_8s"])

max |y_2s - sum of 4 y_500ms|  = 4.3576795817623637e-16
max |y_8s - sum of 16 y_500ms| = 6.527981280535222e-16


**Result:** about 1e-15, which is floating-point rounding. All three targets are built from **one** label-mid
series sampled every 500 ms.

## 2. The 10 ms offset, and information we cannot see

Our quotes live on a 100 ms grid, so the as-of mid at t + 10 ms is the mid at t. Rebuilding labels from
quotes with offset 10 ms or 0 ms gives **identical** results. They match the supplied labels about 98% of
the time. The rest are moments when the real price already moved inside (t, t + 10 ms], which our data
can't show.

In [3]:
days.group_by("instrument").agg(
    pl.col("y_500ms_rebuild_exact_pct").median().alias("rebuild_exact_%"),
    pl.col("y_500ms_rebuild_corr").median().alias("corr_500ms"),
    pl.col("y_8s_rebuild_corr").median().alias("corr_8s"),
    pl.col("label_mid_eq_grid_mid_pct").median().alias("label_mid_==_our_mid_%")).sort("instrument")

instrument,rebuild_exact_%,corr_500ms,corr_8s,label_mid_==_our_mid_%
str,f64,f64,f64,f64
"""1000PEPEUSDT_PERP""",97.160,0.987,0.999,98.527
"""BTCUSDT_PERP""",98.383,0.989,1.000,99.179
"""ETHUSDT_PERP""",97.810,0.988,0.999,98.831
"""SOLUSDT_PERP""",98.750,0.986,0.999,99.336


**What this means:** the supplied labels are ground truth, so we never train on our own rebuild. The hidden
part only matters at 500 ms (correlation 0.99); at 8 s it is diluted to almost nothing (> 0.999).
Note that this is not a leak we could exploit: those 10 ms are after t.

## 3. Positive, negative, zero, and very discrete

Because the spread is almost always one tick, bid and ask move together, so the mid moves in **whole
ticks**. Below, 500 ms moves measured in half-ticks.

In [4]:
cols = [f"y_500ms_pct_{k}_half_ticks" for k in range(5)] + ["y_500ms_pct_gt_4_half_ticks"]
disc = days.group_by("instrument").agg([pl.col(c).median() for c in cols]).sort("instrument")
fig, ax = plt.subplots(figsize=(8, 3.4))
labels = ["0", "1/2", "1", "3/2", "2", ">2"]
w = 0.2
for k, r in enumerate(disc.iter_rows(named=True)):
    ax.bar(np.arange(6) + k * w, [r[c] for c in cols], w, label=r["instrument"])
ax.set_xticks(np.arange(6) + 1.5 * w, labels); ax.set_xlabel("|500 ms mid move| in ticks"); ax.set_ylabel("% of points")
ax.set_yscale("log"); ax.legend(fontsize=7, frameon=False); ax.set_title("500 ms targets are mostly 0 or whole ticks")
fig.tight_layout(); fig.savefig("stage5_discreteness.png", dpi=130)
disc

instrument,y_500ms_pct_0_half_ticks,y_500ms_pct_1_half_ticks,y_500ms_pct_2_half_ticks,y_500ms_pct_3_half_ticks,y_500ms_pct_4_half_ticks,y_500ms_pct_gt_4_half_ticks
str,f64,f64,f64,f64,f64,f64
"""1000PEPEUSDT_PERP""",66.260,0.161,6.319,0.122,5.849,21.420
"""BTCUSDT_PERP""",80.382,0.008,0.716,0.004,0.250,18.544
"""ETHUSDT_PERP""",71.724,0.009,1.092,0.005,0.645,26.724
"""SOLUSDT_PERP""",78.356,0.010,11.339,0.009,4.280,2.282


**Read it:** SOL's tick is coarse (about 1.2 bps). 78% of its 500 ms targets are 0 and 11% are exactly
±1 tick, so it's almost a three-way classification problem dressed as regression. BTC's tick is tiny relative
to its volatility, so when it moves, it usually moves several ticks at once.

**Why it's still regression:** the metric is Pearson correlation with the *size* of the move, not
just its sign. A model that is confident on big moves gains more than one that only gets signs right.

## 4. Fat tails and the official clipping

Clip bounds are the training set's 0.1% and 99.9% quantiles per instrument and horizon. Clipping touches
only 0.2% of points, but look how much of the variance (sum of squares) those points hold:

In [5]:
days.group_by("instrument").agg(
    [pl.col(f"{h}_ss_share_in_tails_pct").median().alias(f"variance_in_tails_%_{h[2:]}") for h in ("y_500ms", "y_2s", "y_8s")]
    + [pl.col("y_500ms_kurtosis").median().alias("kurtosis_raw"), pl.col("y_500ms_kurtosis_clipped").median().alias("kurtosis_clipped")]
).sort("instrument")

instrument,variance_in_tails_%_500ms,variance_in_tails_%_2s,variance_in_tails_%_8s,kurtosis_raw,kurtosis_clipped
str,f64,f64,f64,f64,f64
"""1000PEPEUSDT_PERP""",16.737,11.585,8.485,54.206,16.989
"""BTCUSDT_PERP""",19.698,12.464,8.660,84.303,25.402
"""ETHUSDT_PERP""",19.453,14.083,10.032,80.016,22.203
"""SOLUSDT_PERP""",16.773,11.487,7.520,57.215,18.141


About **17–20% of the 500 ms target's variance sits in 0.2% of the points.** Pearson weights by squared size,
so without clipping a few crash seconds would dominate both training and the score. Rule: train on the
**clipped** target, exactly as the metric uses it.

## 5. Overlapping labels, and the purge

`y_8s(t)` and `y_8s(t + 500 ms)` share 15 of their 16 half-second steps, so they're almost the same number.
Theory for independent steps: autocorrelation = (16 − k)/16 at lag k.

In [6]:
lags = [1, 4, 8, 15, 16, 17, 20]
fig, ax = plt.subplots(figsize=(7, 3.2))
for inst in sorted(days["instrument"].unique()):
    g = days.filter(pl.col("instrument") == inst)
    ax.plot(lags, [g[f"acf8s_lag{L}"].median() for L in lags], "o-", label=inst)
ax.plot(range(0, 21), [max(16 - k, 0) / 16 for k in range(21)], "k--", lw=0.8, label="(16-k)/16 theory")
ax.set_xlabel("lag (500 ms steps)"); ax.set_ylabel("autocorrelation of y_8s"); ax.legend(fontsize=7, frameon=False)
ax.set_title("y_8s overlap dies out after 16 steps = 8 s"); fig.tight_layout(); fig.savefig("stage5_overlap.png", dpi=130)

**Consequences for validation (Stage 10):**
* Rows are not independent. 172,800 rows a day carry far less information than that count suggests, so
  significance tests must work on **days**, not rows.
* Between a training row and a validation row we need ≥ 8.01 s of gap. The metric is per day, so we split by
  **whole UTC days**.
* Labels in the last 8 s of a day use the **next day's** prices (we checked: the last 16 `y_8s` are filled). So
  when day d is in training and d + 1 in validation, the training labels peek into validation. Fix: a 1-day
  embargo, or drop each training day's last 16 rows before a validation day.

## 6. A hint for Stage 7: momentum at 500 ms

In [7]:
days.group_by("instrument").agg([pl.col(f"acf500_lag{L}").median().alias(f"acf_y500_lag{L}") for L in (1, 2, 4)]).sort("instrument")

instrument,acf_y500_lag1,acf_y500_lag2,acf_y500_lag4
str,f64,f64,f64
"""1000PEPEUSDT_PERP""",0.074,0.028,0.010
"""BTCUSDT_PERP""",0.104,0.040,0.017
"""ETHUSDT_PERP""",0.069,0.023,0.010
"""SOLUSDT_PERP""",0.061,0.016,0.007


Consecutive 500 ms targets correlate about +0.06 to +0.10: moves tend to **continue**. That's a *hypothesis*
for a feature, the observable past return over the last 500 ms. We'll measure it properly on training days
in Stage 9. (`y_500ms(t − 500)` itself is NOT observable at t; it needs M(t) = mid at t + 10 ms.)

## 7. Train vs validation: the regime moved

In [8]:
days.group_by("instrument", "split").agg(pl.col("y_500ms_std_bps").mean().alias("std_500ms_bps"),
    pl.col("y_8s_std_bps").mean().alias("std_8s_bps"), pl.col("y_500ms_zero_pct").mean().alias("zero_%")).sort("instrument", "split")

instrument,split,std_500ms_bps,std_8s_bps,zero_%
str,str,f64,f64,f64
"""1000PEPEUSDT_PERP""","""train""",1.064,4.762,62.104
"""1000PEPEUSDT_PERP""","""valid""",0.893,3.986,73.277
"""BTCUSDT_PERP""","""train""",0.504,2.362,77.091
"""BTCUSDT_PERP""","""valid""",0.370,1.731,85.853
"""ETHUSDT_PERP""","""train""",0.762,3.366,68.294
"""ETHUSDT_PERP""","""valid""",0.532,2.344,79.649
"""SOLUSDT_PERP""","""train""",0.863,3.764,77.671
"""SOLUSDT_PERP""","""valid""",0.587,2.560,89.110


Validation is calmer on every perp: 15–32% lower volatility, 9–11 points more zeros. Pearson doesn't care about
scale, but a model that learned "big moves are common" will be mis-calibrated. Prefer features normalised
by recent activity.

## Mastery questions
1. Why can't `y_500ms(t − 500 ms)` be used as a feature at time t, even though it "ends" at t?
2. If day d's last 16 `y_8s` labels use day d + 1's prices, what goes wrong when d is in training and d + 1 in validation?
3. SOL's 500 ms target is mostly {−1 tick, 0, +1 tick}. Is Pearson correlation still a sensible metric? What does
   it reward?

### Answers
1. It is ln M(t) − ln M(t − 500) with M(t) = mid(t + 10 ms), so it needs a price from 10 ms in the future. The
   observable version is our grid mid at t versus t − 500.
2. Training labels contain information about the first 8 s of the validation day, so validation looks
   slightly better than reality. That's leakage through the label.
3. Yes. It rewards ranking the moves correctly within each day, and getting the big ones right counts most.
   It's still sensitive to the few large moves, which is why clipping matters.